# Atelier 3.C — Une équipe d'agents

**Durée : 40 min**

Une même tâche, la réponse à une demande de remboursement, confiée à trois agents spécialisés :

- l'**analyste** consulte le CRM et calcule le montant (outils) ; il produit une fiche d'analyse ;
- le **rédacteur** écrit la réponse au client à partir de la fiche ;
- le **relecteur** vérifie la réponse et la renvoie au rédacteur s'il trouve un problème, deux fois au plus.

C'est le schéma « évaluateur-optimiseur » : un agent produit, un autre critique.

In [1]:
import json
from typing import TypedDict

import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

import agents_fictiva as af
import outils_llm as llm

MODELE = llm.choisir_modele()
print("Modèle :", MODELE)

def formater(demande):
    return f"De : {demande['expediteur']} ({demande['entreprise']})\nObjet : {demande['objet']}\n\n{demande['corps']}"

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Modèle : groq:openai/gpt-oss-20b


## 1. Les trois rôles

L'analyste réutilise le graphe de l'atelier 3.B, avec un autre prompt et **sans** l'outil d'écriture : il n'a pas besoin de créer de ticket. Chaque agent reçoit seulement ce qu'il lui faut.

In [2]:
PROMPT_ANALYSTE = """Tu es l'analyste du service après-vente de Fictiva SA. Nous sommes le 23 octobre 2026.
Avec les outils du CRM, établis une fiche d'analyse de la demande, sans rédiger de réponse au client :
- client (numéro, nom, statut) et commande concernée ;
- ce que demande le client ;
- décision : montant remboursable calculé avec l'outil calculer_remboursement (jamais de tête), motif, validation requise ou non ;
- points d'attention (informations manquantes, contradictions entre l'email et le CRM)."""

PROMPT_REDACTEUR = """Tu rédiges les réponses du service client de Fictiva SA : vouvoiement, ton cordial et professionnel, 8 lignes au plus.
Appuie-toi uniquement sur la fiche d'analyse : ne promets rien qu'elle ne contient pas. Si une validation est requise, dis que la demande est en cours de validation.
Si un relecteur a signalé des problèmes, corrige-les tous."""

PROMPT_RELECTEUR = """Tu relis une réponse du service client de Fictiva SA avant envoi. Vérifie :
1. que les montants, délais et décisions sont exactement ceux de la fiche d'analyse ;
2. qu'aucune promesse ne dépasse la fiche (remboursement, délai, geste commercial) ;
3. que le ton est professionnel et que la réponse traite bien la demande du client.
N'approuve que si tout est correct ; sinon, liste précisément chaque problème."""

class Revue(BaseModel):
    approuve: bool
    problemes: list[str] = Field(description="Problèmes précis à corriger ; liste vide si la réponse est approuvée")

analyste = af.construire_agent(
    [o for o in await af.outils_crm_mcp()] + [af.calculer_remboursement],
    prompt=PROMPT_ANALYSTE, modele=MODELE, memoire=False,
)
redacteur = af.modele_chat(MODELE)
relecteur = af.modele_chat(MODELE).with_structured_output(Revue)

## 2. Le graphe de l'équipe

L'état partagé circule de nœud en nœud. Une arête conditionnelle renvoie au rédacteur tant que le relecteur n'approuve pas, avec une limite de deux tours.

In [3]:
class Etat(TypedDict, total=False):
    demande: str
    fiche: str
    brouillon: str
    revue: dict
    tours: int

async def noeud_analyste(etat: Etat, config):
    resultat = await analyste.ainvoke({"messages": [HumanMessage(etat["demande"])]}, config)
    return {"fiche": resultat["messages"][-1].content, "tours": 0}

async def noeud_redacteur(etat: Etat, config):
    consignes = f"Demande du client :\n{etat['demande']}\n\nFiche d'analyse :\n{etat['fiche']}"
    if etat.get("revue") and not etat["revue"]["approuve"]:
        consignes += "\n\nProblèmes signalés par le relecteur :\n- " + "\n- ".join(etat["revue"]["problemes"])
        consignes += f"\n\nVersion précédente :\n{etat['brouillon']}"
    reponse = await redacteur.ainvoke([SystemMessage(PROMPT_REDACTEUR), HumanMessage(consignes)], config)
    return {"brouillon": reponse.content, "tours": etat.get("tours", 0) + 1}

async def noeud_relecteur(etat: Etat, config):
    revue = await relecteur.ainvoke([SystemMessage(PROMPT_RELECTEUR), HumanMessage(
        f"Fiche d'analyse :\n{etat['fiche']}\n\nRéponse à relire :\n{etat['brouillon']}")], config)
    return {"revue": revue.model_dump()}

def suite(etat: Etat):
    return END if etat["revue"]["approuve"] or etat["tours"] >= 2 else "redacteur"

graphe = StateGraph(Etat)
graphe.add_node("analyste", noeud_analyste)
graphe.add_node("redacteur", noeud_redacteur)
graphe.add_node("relecteur", noeud_relecteur)
graphe.add_edge(START, "analyste")
graphe.add_edge("analyste", "redacteur")
graphe.add_edge("redacteur", "relecteur")
graphe.add_conditional_edges("relecteur", suite, ["redacteur", END])
equipe = graphe.compile()
print(equipe.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	analyste(analyste)
	redacteur(redacteur)
	relecteur(relecteur)
	__end__([<p>__end__</p>]):::last
	__start__ --> analyste;
	analyste --> redacteur;
	redacteur --> relecteur;
	relecteur -.-> __end__;
	relecteur -.-> redacteur;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 3. L'équipe au travail

In [4]:
demandes = af.charger_demandes()
traceur_equipe = af.TraceurLocal()
etat = await equipe.ainvoke({"demande": formater(demandes[2])}, {"callbacks": [traceur_equipe], "recursion_limit": 40})

print("=== FICHE DE L'ANALYSTE\n" + etat["fiche"])
print("\n=== RÉPONSE FINALE (après", etat["tours"], "rédaction(s))\n" + etat["brouillon"])
print("\n=== DERNIÈRE REVUE\n", json.dumps(etat["revue"], ensure_ascii=False, indent=2))

=== FICHE DE L'ANALYSTE
**Fiche d’analyse – Retour de 10 sièges ergonomiques**

| Élément | Détails |
|---------|---------|
| **Client** | Numéro : CL‑10345 <br>Nom : Keller & Fils SA <br>Statut : standard <br>Ville : Fribourg |
| **Commande concernée** | Numéro : CMD‑2026‑0402 <br>Statut de la commande : livrée <br>Date de livraison : 14 oct 2026 <br>Jours depuis la livraison : 9 <br>Références commandées : SE‑700 (10 pièces, 700 CHF/pc) <br>Total HT (après remise 10 %) : 6 300 CHF |
| **Demande du client** | Retourner 10 sièges SE‑700 non déballés, demande de remboursement. |
| **Décision** |  |
| • Montant remboursable | 6 250 CHF (calculé par `calculer_remboursement`) |
| • Motif | Retour sans défaut dans le délai de 30 jours : frais de traitement de 50 CHF déduits. |
| • Validation requise | Oui (montant > 5 000 CHF) |
| • Articles concernés par la politique | 1.1 (retour sans défaut), 3 (délai 30 jours), 5 (frais de traitement) |
| **Points d’attention** |  |
| 1. **Discrepance d

/opt/anaconda3/lib/python3.13/site-packages/pydantic/main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Revue(approuve=True, problemes=[]), input_type=Revue])
  return self.__pydantic_serializer__.to_python(


## 4. Une équipe, ou un seul agent ?

Même demande, traitée par l'agent unique de l'atelier 3.B (sans création de ticket). On compare le nombre d'appels, les tokens et la durée.

In [5]:
agent_seul = af.construire_agent([o for o in await af.outils_crm_mcp()] + [af.calculer_remboursement],
                                 prompt=af.PROMPT_SAV.replace("3. Crée un ticket qui résume la demande, la décision et le montant éventuel.\n", ""),
                                 modele=MODELE, memoire=False)
traceur_seul = af.TraceurLocal()
resultat = await agent_seul.ainvoke({"messages": [HumanMessage(formater(demandes[2]))]}, {"callbacks": [traceur_seul]})
print(resultat["messages"][-1].content)

pd.DataFrame({"agent seul": traceur_seul.bilan(), "équipe de trois": traceur_equipe.bilan()})

Monsieur Keller,

Suite à votre demande, nous avons calculé le montant remboursable de votre retour de 10 sièges SE‑700, soit 6 250,00 CHF. Ce montant est soumis à validation par un responsable conformément à notre politique, et vous serez informé dès que le processus sera finalisé.  

Le siège étant non déballé et retourné dans le délai de 30 jours, les frais de traitement de 50,00 CHF seront déduits comme indiqué dans les conditions de retour.

Vous recevrez d’ici peu nos instructions concernant l’expédition du matériel vers nos entrepôts et la procédure de remboursement.

Cordialement,  
[Nom du service après‑vente]  
Fictiva SA


,agent seul,équipe de trois
appels_modele,4.0,6.0
appels_outils,3.0,3.0
erreurs,0.0,0.0
tokens_entree,4053.0,5687.0
tokens_sortie,758.0,3161.0
duree_totale_s,3.2,7.2


**Questions**
1. La réponse de l'équipe est-elle meilleure ? Sur quels points le relecteur est-il intervenu ?
2. Combien coûte ce gain, en appels de modèle et en tokens ?
3. Pour quel type de demandes l'équipe vaut-elle son coût ?

✏️ Faites traiter par l'équipe la demande de Golay Architectes (`demandes[5]`), dont la commande n'a jamais été livrée. Le relecteur bloque-t-il une promesse de remboursement ?

## À retenir

- Plusieurs agents spécialisés, chacun avec **son prompt et ses seuls outils**, rendent chaque rôle plus simple et plus contrôlable.
- Le schéma évaluateur-optimiseur (produire, critiquer, corriger) améliore la qualité au prix d'appels supplémentaires.
- Toujours une **limite de tours** : deux agents peuvent se renvoyer la balle indéfiniment.
- On ne passe au multi-agents que si la mesure montre un gain face à un agent unique bien conçu.